<a href="https://colab.research.google.com/github/HariPrasad017/IIP_AIMLTN_ASSESMENT/blob/main/Visualizing_Performance_(ROC_AUC)_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    precision_score, recall_score,
    f1_score, roc_auc_score
)

# Step 2: Create the Dataset

In [2]:
np.random.seed(42)
n = 200

data = {
    'tenure':          np.random.randint(1, 72, n),
    'monthly_charges': np.round(np.random.uniform(20, 120, n), 2),
    'num_services':    np.random.randint(1, 8, n),
    'contract_type':   np.random.randint(0, 3, n),
}
df = pd.DataFrame(data)

log_odds = (
    -0.05 * df['tenure']
    + 0.03 * df['monthly_charges']
    - 0.2  * df['num_services']
    - 0.8  * df['contract_type']
    + np.random.randn(n) * 0.5
)
prob = 1 / (1 + np.exp(-log_odds))
df['churn'] = (prob > 0.65).astype(int)

# Step 3: Split Features & Target

In [3]:
X = df.drop('churn', axis=1)
y = df['churn']

# Step 4: Train-Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 5: Feature Scaling

In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 6: Train Logistic Regression Model

In [6]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

LogisticRegression()

# TASK 1 — Accuracy & Baseline

In [7]:
train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
test_acc = accuracy_score(y_test, model.predict(X_test_scaled))

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

# Naive baseline (majority class)
baseline = y.value_counts(normalize=True).max()
print("Baseline Accuracy:", baseline)

Train Accuracy: 0.95625
Test Accuracy: 0.9
Baseline Accuracy: 0.855


If test accuracy is significantly higher than baseline,
then model performs meaningfully better than naive prediction.

# TASK 2 — Confusion Matrix & Metrics

In [8]:
y_pred = model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

# Extract TN, FP, FN, TP
tn, fp, fn, tp = cm.ravel()
print("TN:", tn, "FP:", fp, "FN:", fn, "TP:", tp)

Confusion Matrix:
 [[34  0]
 [ 4  2]]
Precision: 1.0
Recall: 0.3333333333333333
F1 Score: 0.5
TN: 34 FP: 0 FN: 4 TP: 2


False Negative (FN) is more costly because missing a churn customer


means losing that customer without intervention.

# TASK 3 — Threshold Tuning

In [10]:
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Threshold = 0.5
y_pred_05 = (y_prob >= 0.5).astype(int)
print("Threshold 0.5 Precision:", precision_score(y_test, y_pred_05))
print("Threshold 0.5 Recall:", recall_score(y_test, y_pred_05))

# Threshold = 0.3
y_pred_03 = (y_prob >= 0.3).astype(int)
print("Threshold 0.3 Precision:", precision_score(y_test, y_pred_03))
print("Threshold 0.3 Recall:", recall_score(y_test, y_pred_03))

Threshold 0.5 Precision: 1.0
Threshold 0.5 Recall: 0.3333333333333333
Threshold 0.3 Precision: 1.0
Threshold 0.3 Recall: 1.0


Lowering threshold increases recall but may reduce precision.

Threshold 0.3 is better for churn business since catching more churners is important.

# TASK 4 — ROC-AUC

In [11]:
roc_auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC Score:", roc_auc)

ROC-AUC Score: 1.0


ROC-AUC > 0.5 means model performs better than random guessing.

Closer to 1 indicates strong classification performance.